In [1]:
!pip install ~/fwiVis/utility_functions/
! pip install plotnine


Processing /home/jovyan/fwiVis/utility_functions
  Preparing metadata (setup.py) ... done
  Created wheel for fwiVis: filename=fwiVis-0.1-py3-none-any.whl size=17985 sha256=fd35201dd665e83be02deba69f5b4abfdf70f9e79beb338f272a55413449a68b
  Stored in directory: /tmp/pip-ephem-wheel-cache-n3w3d0hn/wheels/79/0f/3d/08c18473dd7e0fb915900e6b4f13b81f1fa84371f9bf24d864
Successfully built fwiVis
  Using cached plotnine-0.14.5-py3-none-any.whl.metadata (9.3 kB)
  Using cached mizani-0.13.1-py3-none-any.whl.metadata (4.7 kB)
Using cached plotnine-0.14.5-py3-none-any.whl (1.3 MB)
Using cached mizani-0.13.1-py3-none-any.whl (127 kB)


In [2]:
import s3fs
s3 = s3fs.S3FileSystem(anon=False)
from math import cos, asin, sqrt
import re

import numpy as np
import geopandas as gpd
import pandas as pd
from matplotlib import pyplot as plt
import os
import rioxarray as rio
import xarray as xr
import rasterio
import glob
from shapely.errors import ShapelyDeprecationWarning
from shapely.geometry import Point
import warnings
import folium
import datetime
import time
from folium import plugins
warnings.filterwarnings("ignore", category=ShapelyDeprecationWarning) 
warnings.filterwarnings("ignore", category=ShapelyDeprecationWarning) 
#import contextily as cx
from shapely.geometry import box
import sys
from datetime import datetime, timedelta
from itertools import chain

from datetime import date
from bs4 import BeautifulSoup
import requests
import os
import plotnine
import xarray as xr

#import numpy as np
from matplotlib import pyplot as plt
from plotnine import ggplot, geom_point, geom_jitter, aes, stat_smooth, facet_wrap
import plotnine as plotnine
import seaborn as sns

from scipy import stats
from patsy import ModelDesc
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf

import math

import fwiVis.fwiVis as fv

In [44]:
temp = fv.prep_fire_files("~/fwiVis/notebooks/data/with_lat_lon_Quebec_v3_full_data_perimeters20241112.csv", "4326")
#temp = gpd.read_file("~/fwiVis/notebooks/data/with_lat_lon_Quebec_v3_full_data_perimeters20241112.csv")
#temp = temp.to_crs(4326)

In [33]:
temp[['fireID', 't', 'geometry',
       'mergeid', 'ftype', 'n_pixels', 'n_newpixels', 'farea', 'fperim',
       'flinelen', 'duration', 'pixden', 'meanFRP', 't_st', 't_ed',
       'isignition', 't_inactive', 'isactive', 'isdead', 'mayreactivate',
       'geom_counts', 'low_confidence_grouping', 'region', 'primarykey', 'GEOS-5.IMERGEARLY', 'FWI', 'FWI_lead_1', 'FWI_lead_2',
       'FWI_lead_3', 'FWI_lead_4', 'FWI_lead_5', 'FWI_lead_6', 'FWI_lead_7',
       'FWI_lead_8', 'lon_centroid', 'lat_centroid']].to_file('with_lat_lon_Quebec_v3_full_data_perimeters20241112.shp')


/tmp/ipykernel_1299/1531082705.py:7: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
/srv/conda/envs/notebook/lib/python3.12/site-packages/pyogrio/raw.py:723: RuntimeWarning: Normalized/laundered field name: 'n_newpixels' to 'n_newpixel'
/srv/conda/envs/notebook/lib/python3.12/site-packages/pyogrio/raw.py:723: RuntimeWarning: Normalized/laundered field name: 'mayreactivate' to 'mayreactiv'
/srv/conda/envs/notebook/lib/python3.12/site-packages/pyogrio/raw.py:723: RuntimeWarning: Normalized/laundered field name: 'geom_counts' to 'geom_count'
/srv/conda/envs/notebook/lib/python3.12/site-packages/pyogrio/raw.py:723: RuntimeWarning: Normalized/laundered field name: 'low_confidence_grouping' to 'low_confid'
/srv/conda/envs/notebook/lib/python3.12/site-packages/pyogrio/raw.py:723: RuntimeWarning: Normalized/laundered field name: 'GEOS-5.IMERGEARLY' to 'GEOS-5.IME'
/srv/conda/envs/notebook/lib/python3.12/site-packages/pyogrio/raw.py:723: Runt

In [46]:
tmp = temp[temp.fireID == '226']
tmp = tmp[['fireID', 't', 'geometry',
       'mergeid', 'ftype', 'n_pixels', 'n_newpixels', 'farea', 'fperim',
       'flinelen', 'duration', 'pixden', 'meanFRP', 't_st', 't_ed',
       'isignition', 't_inactive', 'isactive', 'isdead', 'mayreactivate',
       'geom_counts', 'low_confidence_grouping', 'region', 'primarykey',
        'GEOS-5.IMERGEARLY', 'FWI', 'FWI_lead_1', 'FWI_lead_2',
       'FWI_lead_3', 'FWI_lead_4', 'FWI_lead_5', 'FWI_lead_6', 'FWI_lead_7',
       'FWI_lead_8', 'lon_centroid', 'lat_centroid']]

#tmp.explore()

In [51]:
lf = gpd.read_file("/home/jovyan/fireatlast_nrt/fireatlas/data/FEDSoutput-v3/Quebec_V3/2023/CombinedLargefire/20230915PM/lf_fireline.fgb")

In [58]:
lf.fireID = lf.fireID.astype("int")
lf.fireID = lf.fireID.astype("str")

lf = lf.to_crs(4326)

In [8]:
lf.to_file("~/fwiVis/notebooks/data/firelines.shp")

/srv/conda/envs/notebook/lib/python3.12/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field t_st create as date field, though DateTime requested.
/srv/conda/envs/notebook/lib/python3.12/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field t_ed create as date field, though DateTime requested.


In [ ]:
tmp_lf = lf[lf.fireID == '226']
tmp_lf = tmp_lf.sort_values(by = "t")

In [104]:
m = tmp[2:3].explore()

In [105]:
tmp_lf[2:3].explore(m = m, color = "orange")